<a href="https://colab.research.google.com/github/AkramMoustafa/air_pollution_prediction/blob/main/AirPredictionModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#aws s3 cp \
###--no-sign-request \
#--recursive \
#s3://openaq-data-archive/records/csv.gz/locationid=2178/year=2020/ \
#data

# This is the yearly data pulling through the AWS Bucket


In [ ]:
!pip -q install "numpy>=1.26,<2.1" requests pandas pyarrow haversine scikit-learn xgboost scipy arcgis arcgis-mapping boto3

# **INSTALL DEPENDENCIES AND VALUES**

In [3]:
# @title
import os
import time
import csv
import random
import json
import hashlib
import requests
import pandas as pd
import numpy as np

import io
import gzip

from datetime import datetime, timedelta, timezone
from google.colab import userdata

# Distance / neighbors
from sklearn import neighbors
from scipy.spatial.distance import pdist, squareform

# HTTP session/retries
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# AWS OpenAQ archive
import boto3
from botocore import UNSIGNED
from botocore.client import Config

# ArcGIS
from arcgis.gis import GIS
from arcgis.features import GeoAccessor, GeoSeriesAccessor

# ModeL imports
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

In [ ]:
API_KEY = userdata.get("OpenAQKey")
if not API_KEY:
    raise ValueError("Missing OpenAQ API key. Add it in Colab secrets as 'OpenAQKey'.")

ARC_KEY = userdata.get("ArcGISKey")
if not ARC_KEY:
    raise ValueError("Missing ArcGIS API Key. Add it in Colab secrets as 'ArcGISKey'")

In [ ]:
# Key Variables
LAT, LON = 34.1808, -118.3090
RADIUS_M = 6_000

CORE_PARAMS = {"pm25", "pm2.5", "pm10"}
INCLUDE_ALL_PARAMS = False

DAYS = 365 * 3
CHUNK_DAYS = 30

LOCATION_LIMIT = 100
MEAS_LIMIT = 1000
MAX_PAGES_LOC = 50
MAX_PAGES_MEAS = 50

# Backfill AWS
USE_AWS_BACKFILL = True
ALLOW_API_FALLBACK = False
AWS_BUCKET = "openaq-data-archive"
AWS_PREFIX = "records/csv.gz"

BASE_SLEEP = 1.00
JITTER = 0.2

CACHE_TTL_HOURS = 24
CACHE_ENABLED = True

BASE_URL = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": API_KEY}

In [ ]:
RUN_TAG = f"openaq_{LAT}_{LON}_r{RADIUS_M}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT_DIR = RUN_TAG

BIG_CSV_PATH = os.path.join(OUT_DIR, "all_measurements_long.csv")
SENSORS_CSV_PATH = os.path.join(OUT_DIR, "sensors_in_radius.csv")
FAIL_CSV_PATH = os.path.join(OUT_DIR, "failures.csv")
BAD_ROWS_PATH = os.path.join(OUT_DIR, "bad_datetime_rows.csv")

SENSOR_MAP = os.path.join(OUT_DIR, "sensor_map.html")
ARC_SENSOR_MAP = os.path.join(OUT_DIR, "sensor_map_arcgis.html")
DIST_PATH = os.path.join(OUT_DIR, "sensor_distances_matrix.csv")

PART_DIR = os.path.join(OUT_DIR, "partitioned")
TRAIN_DIR = os.path.join(OUT_DIR, "train_partitioned")
TEST_DIR = os.path.join(OUT_DIR, "test_partitioned")

PARQUET_FULL_DIR = os.path.join(OUT_DIR, "parquet_full_chunks")
PARQUET_TRAIN_DIR = os.path.join(OUT_DIR, "parquet_train_chunks")
PARQUET_TEST_DIR = os.path.join(OUT_DIR, "parquet_test_chunks")
PARQUET_BAD_DIR = os.path.join(OUT_DIR, "parquet_bad_rows")

for d in [
    OUT_DIR, PART_DIR, TRAIN_DIR, TEST_DIR,
    PARQUET_FULL_DIR, PARQUET_TRAIN_DIR, PARQUET_TEST_DIR, PARQUET_BAD_DIR
]:
    os.makedirs(d, exist_ok=True)

In [ ]:
session = requests.Session()

retry_cfg = Retry(
    total=6,
    read=6,
    connect=6,
    backoff_factor=0.7,
    status_forcelist=[408, 429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry_cfg, pool_connections=50, pool_maxsize=50)
session.mount("https://", adapter)
session.mount("http://", adapter)

REQUEST_COUNT = 0
RATE_LIMIT_COUNT = 0
RETRYABLE_ERROR_COUNT = 0
HTTP_ERROR_COUNT = 0
CACHE_HITS = 0

CACHE_PATH = os.path.join(OUT_DIR, "api_cache.json")
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, "r", encoding="utf-8") as f:
        RESPONSE_CACHE = json.load(f)
else:
    RESPONSE_CACHE = {}

s3_client = boto3.client("s3", config=Config(signature_version=UNSIGNED))

# **File Writing and API Help**

In [ ]:
def normalize_param(p):
    if p is None:
        return None
    p = str(p).strip().lower()
    if p == "pm2.5":
        return "pm25"
    return p

def sleep_time_api(mult=1.0):
    time.sleep((BASE_SLEEP + random.random() * JITTER) * mult)

def sleep_time_aws(mult=1.0):
    time.sleep(.1)

def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

def iteration_chunks(start_dt, end_dt, chunk_days=30):
    cur = start_dt
    while cur < end_dt:
        next_dt = min(cur + timedelta(days=chunk_days), end_dt)
        yield cur, next_dt
        cur = next_dt

def cache_key(endpoint, params):
    payload = {"endpoint": endpoint, "params": params}
    raw = json.dumps(payload, sort_keys=True)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def cache_valid(entry):
    if not entry:
        return False
    ts = pd.to_datetime(entry.get("cached_at"), errors="coerce", utc=True)
    if pd.isna(ts):
        return False
    age_h = (pd.Timestamp.now(tz="UTC") - ts).total_seconds() / 3600
    return age_h <= CACHE_TTL_HOURS

def flush_cache():
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(RESPONSE_CACHE, f)

In [ ]:
def get_json(endpoint, params=None, retries=6):
    global REQUEST_COUNT, RATE_LIMIT_COUNT, RETRYABLE_ERROR_COUNT, HTTP_ERROR_COUNT,CACHE_HITS

    url = f"{BASE_URL}{endpoint}"
    params = params or {}

    ck = cache_key(endpoint, params)
    if CACHE_ENABLED:
        entry = RESPONSE_CACHE.get(ck)
        if cache_valid(entry):
            CACHE_HITS += 1
            return entry["payload"]

    backoff = 1.0

    for attempt in range(retries):
        try:
            REQUEST_COUNT += 1
            r = session.get(url, headers=HEADERS, params=params, timeout=60)
            print(f"[HTTP] {r.status_code} endpoint={endpoint} attempt={attempt+1}/{retries}")

            remaining = r.headers.get("x-ratelimit-remaining")
            reset_ts = r.headers.get("x-ratelimit-reset")
            if remaining is not None or reset_ts is not None:
                print(f"[HTTP] rate-limit remaining={remaining} reset={reset_ts}")

            if r.status_code == 429:
                RATE_LIMIT_COUNT += 1
                RETRYABLE_ERROR_COUNT += 1
                if reset_ts:
                    wait_s = max(1, int(reset_ts) - int(time.time()) + 1)
                    print(f"[HTTP] 429 hit, sleeping {wait_s}s")
                    time.sleep(wait_s)
                else:
                    print(f"[HTTP] 429 hit, backoff={backoff:.2f}")
                    sleep_time_api(mult=backoff)
                    backoff *= 2
                continue

            if r.status_code in (408, 500, 502, 503, 504):
                RETRYABLE_ERROR_COUNT += 1
                print(f"[HTTP] retryable status={r.status_code}, backoff={backoff:.2f}")
                sleep_time_api(mult=backoff)
                backoff *= 2
                continue

            if r.status_code in (401, 403, 404, 422):
                HTTP_ERROR_COUNT += 1
                r.raise_for_status()

            r.raise_for_status()
            payload = r.json()
            if CACHE_ENABLED:
                RESPONSE_CACHE[ck] = {
                    "cached_at": pd.Timestamp.now(tz="UTC").isoformat(),
                    "payload": payload
                }
                if len(RESPONSE_CACHE) % 25 == 0:
                    flush_cache()

            return payload

        except requests.exceptions.RequestException as e:
            print(f"[HTTP] request exception endpoint={endpoint}: {repr(e)}")
            if attempt == retries - 1:
                raise
            RETRYABLE_ERROR_COUNT += 1
            sleep_time_api(mult=backoff)
            backoff *= 2

    raise RuntimeError("Unreachable")


def paginate(endpoint, params, limit=1000, max_pages=50):
    out = []
    last_meta_found = None

    for page in range(1, max_pages + 1):
        p = dict(params)
        p["limit"] = limit
        p["page"] = page

        data = get_json(endpoint, p)
        results = data.get("results", [])
        meta = data.get("meta", {}) or {}
        last_meta_found = meta.get("found")

        print(f"[PAGINATE] endpoint={endpoint} page={page} rows_this_page={len(results)}")

        if last_meta_found is not None:
            print(f"[PAGINATE] endpoint={endpoint} meta.found={last_meta_found}")

        if not results:
            break

        out.extend(results)

        if len(results) < limit:
            break

        sleep_time_api()

    if last_meta_found is not None and len(out) < last_meta_found:
        print(f"[PAGINATE WARNING] endpoint={endpoint} pulled {len(out)} of {last_meta_found}")

    print(f"[PAGINATE DONE] endpoint={endpoint} total_rows={len(out)}")
    return out

In [ ]:
BIG_HEADER = [
    "sensor_id", "parameter", "value",
    "datetime_utc", "datetime_local",
    "location_id", "location_name", "location_lat", "location_lon",
    "sensor_name", "units",
    "lag_1h", "lag_6h", "lag_12h", "lag_24h",
    "neighbor_avg_1h", "neighbor_weighted_1h"
]

def append_rows_to_csv(path, header, rows):
    exists = os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if not exists:
            w.writerow(header)
        w.writerows(rows)

def write_parquet_chunk(df, base_dir, chunk_label):
    if df.empty:
        return
    df.to_parquet(
        os.path.join(base_dir, f"{chunk_label}.parquet"),
        engine="pyarrow",
        compression="snappy",
        index=False
    )

def write_partition_csv(df, base_dir):
    if df.empty:
        return

    df = df.copy()
    df["dtp"] = pd.to_datetime(df["datetime_utc"], errors="coerce", utc=True)
    df["year"] = df["dtp"].dt.year.fillna(-1).astype(int)
    df["month"] = df["dtp"].dt.month.fillna(-1).astype(int)

    for (param, sid, y, m), g in df.groupby(["parameter", "sensor_id", "year", "month"]):
        ystr = "unknown" if y == -1 else f"{y:04d}"
        mstr = "unknown" if m == -1 else f"{m:02d}"
        d = os.path.join(base_dir, f"parameter={param}", f"year={ystr}", f"month={mstr}")
        os.makedirs(d, exist_ok=True)
        p = os.path.join(d, f"sensor_{sid}.csv")
        append_rows_to_csv(p, BIG_HEADER, g[BIG_HEADER].values.tolist())

In [ ]:
print("\n" + "="*70)
print("OPENAQ INGESTION PIPELINE")
print("="*70)
print(f"Run tag            : {RUN_TAG}")
print(f"Output directory   : {OUT_DIR}")
print(f"Query center       : lat={LAT}, lon={LON}")
print(f"Radius (m)         : {RADIUS_M:,}")
print(f"Days back          : {DAYS}")
print(f"Chunk days         : {CHUNK_DAYS}")
print(f"Location limit     : {LOCATION_LIMIT}")
print(f"Measurement limit  : {MEAS_LIMIT}")
print(f"Max pages loc      : {MAX_PAGES_LOC}")
print(f"Max pages meas     : {MAX_PAGES_MEAS}")
print(f"Include all params : {INCLUDE_ALL_PARAMS}")
print(f"Core params        : {sorted(CORE_PARAMS)}")
print("="*70 + "\n")

dt_to = datetime.now(timezone.utc)
dt_from = dt_to - timedelta(days=DAYS)

with open(BIG_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(BIG_HEADER)


OPENAQ INGESTION PIPELINE
Run tag            : openaq_34.1808_-118.309_r6000_20260324_215006
Output directory   : openaq_34.1808_-118.309_r6000_20260324_215006
Query center       : lat=34.1808, lon=-118.309
Radius (m)         : 6,000
Days back          : 1095
Chunk days         : 30
Location limit     : 100
Measurement limit  : 1000
Max pages loc      : 50
Max pages meas     : 50
Include all params : False
Core params        : ['pm10', 'pm2.5', 'pm25']



# **Get Locations, Sensor Table, and Sensor Grouping**

In [ ]:
locations = paginate(
    "/locations",
    params={"coordinates": f"{LAT},{LON}", "radius": RADIUS_M},
    limit=LOCATION_LIMIT,
    max_pages=MAX_PAGES_LOC
)

print("\n" + "-"*60)
print("LOCATION SUMMARY")
print("-"*60)
print(f"Locations found: {len(locations)}")
if locations:
    sample_loc = locations[0]
    print(f"Sample location ID   : {sample_loc.get('id')}")
    print(f"Sample location name : {sample_loc.get('name')}")
    print(f"Sample coordinates   : {sample_loc.get('coordinates')}")
print("-"*60 + "\n")

[HTTP] 200 endpoint=/locations attempt=1/6
[HTTP] rate-limit remaining=59 reset=60
[PAGINATE] endpoint=/locations page=1 rows_this_page=4
[PAGINATE] endpoint=/locations meta.found=4
[PAGINATE DONE] endpoint=/locations total_rows=4

------------------------------------------------------------
LOCATION SUMMARY
------------------------------------------------------------
Locations found: 4
Sample location ID   : 8236
Sample location name : North Holywood
Sample coordinates   : {'latitude': 34.181977, 'longitude': -118.363036}
------------------------------------------------------------



In [ ]:
sensor_rows = []
param_set = set()

for loc in locations:
    loc_coords = loc.get("coordinates") or {}
    for s in (loc.get("sensors") or []):
        param = s.get("parameter") or {}
        raw_name = param.get("name") if isinstance(param, dict) else None
        p_name = normalize_param(raw_name)

        if p_name:
            param_set.add(p_name)

        sensor_rows.append({
            "sensor_id": s.get("id"),
            "sensor_name": s.get("name"),
            "parameter": p_name,
            "units": (param.get("units") if isinstance(param, dict) else None),
            "location_id": loc.get("id"),
            "location_name": loc.get("name"),
            "location_lat": loc_coords.get("latitude"),
            "location_lon": loc_coords.get("longitude"),
        })

df_sensors = pd.DataFrame(sensor_rows).drop_duplicates("sensor_id").reset_index(drop=True)
df_sensors.to_csv(SENSORS_CSV_PATH, index=False)

print("\n" + "-"*60)
print("SENSOR SUMMARY")
print("-"*60)
print(f"Total sensors found: {len(df_sensors)}")
if not df_sensors.empty:
    print("Sensor counts by parameter:")
    print(df_sensors["parameter"].value_counts(dropna=False).to_string())
    print("\nSample sensors:")
    print(df_sensors.head(10).to_string(index=False))
print("-"*60 + "\n")

if df_sensors.empty:
    raise ValueError("No sensors found in radius. Try different LAT/LON/RADIUS.")


------------------------------------------------------------
SENSOR SUMMARY
------------------------------------------------------------
Total sensors found: 20
Sensor counts by parameter:
parameter
temperature         5
pm25                4
pm1                 3
pm10                2
no                  1
no2                 1
nox                 1
o3                  1
relativehumidity    1
um003               1

Sample sensors:
 sensor_id   sensor_name   parameter units  location_id    location_name  location_lat  location_lon
   4272409        no ppm          no   ppm         8236   North Holywood     34.181977   -118.363036
     24002       no2 ppm         no2   ppm         8236   North Holywood     34.181977   -118.363036
   4272242       nox ppm         nox   ppm         8236   North Holywood     34.181977   -118.363036
     24001        o3 ppm          o3   ppm         8236   North Holywood     34.181977   -118.363036
     24000    pm25 µg/m³        pm25 µg/m³         8236   

In [ ]:
normalized_core_params = {normalize_param(x) for x in CORE_PARAMS}

if INCLUDE_ALL_PARAMS:
    params_to_pull = sorted([p for p in param_set if p])
else:
    params_to_pull = sorted([p for p in param_set if p in normalized_core_params])

if not params_to_pull:
    print("No params found (pm25/pm10). Fall back to all params.")
    params_to_pull = sorted([p for p in param_set if p])

print("\n" + "-"*60)
print("FILTERED SUMMARY")
print("-"*60)
print(f"Parameters to pull: {params_to_pull}")

df_sensors_use = df_sensors[df_sensors["parameter"].isin(params_to_pull)].copy()

sensor_ids = df_sensors_use["sensor_id"].dropna().astype(int).tolist()

print(f"Sensors used      : {len(sensor_ids)}")
if not df_sensors_use.empty:
    print("Sensor counts by parameter:")
    print(df_sensors_use["parameter"].value_counts(dropna=False).to_string())
    print("\nFirst few filtered sensors:")
    print(df_sensors_use.head(10).to_string(index=False))
print("-"*60 + "\n")

if not sensor_ids:
    raise ValueError("No sensors for parameters. Try INCLUDE_ALL_PARAMS=True or a different area.")

meta = df_sensors_use.set_index("sensor_id").to_dict(orient="index")


------------------------------------------------------------
FILTERED SUMMARY
------------------------------------------------------------
Parameters to pull: ['pm10', 'pm25']
Sensors used      : 6
Sensor counts by parameter:
parameter
pm25    4
pm10    2

First few filtered sensors:
 sensor_id sensor_name parameter units  location_id         location_name  location_lat  location_lon
     24000  pm25 µg/m³      pm25 µg/m³         8236        North Holywood     34.181977   -118.363036
   2000572  pm10 µg/m³      pm10 µg/m³       947177      Oxnard ES (5918)     34.178500   -118.368650
   2000565  pm25 µg/m³      pm25 µg/m³       947177      Oxnard ES (5918)     34.178500   -118.368650
   2001010  pm10 µg/m³      pm10 µg/m³       947247 Toluca Lake ES (7192)     34.159210   -118.360420
   2001033  pm25 µg/m³      pm25 µg/m³       947247 Toluca Lake ES (7192)     34.159210   -118.360420
  14152560  pm25 µg/m³      pm25 µg/m³      3778909               Burbank     34.171403   -118.310135


In [ ]:
coords = df_sensors_use[
    ["sensor_id", "location_lat", "location_lon"]
].dropna().drop_duplicates("sensor_id").reset_index(drop=True)

lat_lon = np.radians(coords[["location_lat", "location_lon"]].values)

def haversine_vectorized(u, v):
    dlat = v[0] - u[0]
    dlon = v[1] - u[1]
    a = np.sin(dlat/2)**2 + np.cos(u[0]) * np.cos(v[0]) * np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

dist_matrix = squareform(pdist(lat_lon, metric=haversine_vectorized))

df_distances = pd.DataFrame(
    dist_matrix,
    index=coords.sensor_id,
    columns=coords.sensor_id
)

df_distances.to_csv(DIST_PATH)
print(f"[DISTANCES] Saved matrix: {DIST_PATH}")

K_NEIGHBORS = 3
neighbors_map = {}

for sid in df_distances.index:
    nearest = (
        df_distances.loc[sid]
        .sort_values()
        .iloc[1:K_NEIGHBORS+1]
    )
    neighbors_map[sid] = list(zip(nearest.index, nearest.values))

print(f"[NEIGHBORS] Built neighbor map with k={K_NEIGHBORS}")

[DISTANCES] Saved matrix: openaq_34.1808_-118.309_r6000_20260324_215006/sensor_distances_matrix.csv
[NEIGHBORS] Built neighbor map with k=3


# **History Backfill**

In [ ]:
failures = []
sensor_time_min = {}
sensor_time_max = {}

def update_minmax(sid, dt_utc):
    if pd.isna(dt_utc):
        return
    mn = sensor_time_min.get(sid)
    mx = sensor_time_max.get(sid)
    if mn is None or dt_utc < mn:
        sensor_time_min[sid] = dt_utc
    if mx is None or dt_utc > mx:
        sensor_time_max[sid] = dt_utc

pull_start_dt = datetime.now(timezone.utc) - timedelta(days=DAYS)
pull_end_dt = datetime.now(timezone.utc)

In [ ]:
def aws_fetch_sensor_hours(location_id, sensor_id, parameter, y_start, y_end):

    if location_id is None:
        return pd.DataFrame()

    year = y_start.year
    prefix = f"{AWS_PREFIX}/locationid={int(location_id)}/year={year}/"

    out_frames = []

    paginator = s3_client.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=AWS_BUCKET, Prefix=prefix)

    for page in pages:
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.endswith(".csv.gz"):
                continue

            try:
                body = s3_client.get_object(Bucket=AWS_BUCKET, Key=key)["Body"].read()
                with gzip.GzipFile(fileobj=io.BytesIO(body)) as gz:
                    df = pd.read_csv(gz)
            except Exception as e:
                print(f"[AWS READ FAIL] {key} :: {e}")
                continue

            cols_lower = {c.lower(): c for c in df.columns}

            dt_col = cols_lower.get("datetime") or cols_lower.get("datetime_utc")
            if dt_col is None or "value" not in cols_lower:
                continue

            val_col = cols_lower["value"]
            df["datetime_utc"] = pd.to_datetime(df[dt_col], errors="coerce", utc=True)

            param_col = (
                cols_lower.get("parameter")
                or cols_lower.get("parameter_name")
                or cols_lower.get("parametername")
            )
            if param_col is not None:
                df["parameter"] = df[param_col].astype(str).str.lower().map(normalize_param)
            else:
                df["parameter"] = normalize_param(parameter)

            sid_col = (
                cols_lower.get("sensor_id")
                or cols_lower.get("sensorid")
                or cols_lower.get("sensors_id")
            )
            if sid_col is not None:
                df = df[df[sid_col] == sensor_id]

            lid_col = cols_lower.get("location_id") or cols_lower.get("locationid")
            if lid_col is not None:
                df = df[df[lid_col] == int(location_id)]

            df = df[df["parameter"] == normalize_param(parameter)]
            df = df[(df["datetime_utc"] >= y_start) & (df["datetime_utc"] < y_end)]

            if df.empty:
                continue

            out = pd.DataFrame({
                "sensor_id": sensor_id,
                "parameter": df["parameter"],
                "value": pd.to_numeric(df[val_col], errors="coerce"),
                "datetime_utc": df["datetime_utc"],
                "datetime_local": pd.NA
            }).dropna(subset=["datetime_utc", "value"])

            if not out.empty:
                out_frames.append(out)

    if not out_frames:
        return pd.DataFrame()

    return pd.concat(out_frames, ignore_index=True)


for i, sid in enumerate(sensor_ids, 1):
    s_meta = meta.get(sid, {})
    s_param = normalize_param(s_meta.get("parameter")) or "unknown"

    print("=" * 60)
    print(f"[SENSOR {i}/{len(sensor_ids)}] sensor_id={sid}")
    print(f"parameter     : {s_param}")
    print(f"location_id   : {s_meta.get('location_id')}")
    print(f"location_name : {s_meta.get('location_name')}")
    print("=" * 60)

    start_year = pull_start_dt.year
    end_year = pull_end_dt.year

    for y in range(start_year, end_year + 1):
        y_start = max(datetime(y, 1, 1, tzinfo=timezone.utc), pull_start_dt)
        y_end = min(datetime(y + 1, 1, 1, tzinfo=timezone.utc), pull_end_dt)

    print(f"[YEAR START] sensor={sid} from={iso(y_start)} to={iso(y_end)}")

    for y_start, y_end in iteration_chunks(pull_start_dt, pull_end_dt, CHUNK_DAYS):
        print(f"[CHUNK START] sensor={sid} from={iso(y_start)} to={iso(y_end)}")
        try:
            df_chunk = pd.DataFrame()
            loc_id = s_meta.get("location_id")

            if USE_AWS_BACKFILL:
                df_chunk = aws_fetch_sensor_hours(loc_id,sid, s_param, y_start, y_end)
                if not df_chunk.empty:
                    print(f"[AWS BACKFILL] sensor={sid} rows={len(df_chunk)}")

            if df_chunk.empty and ALLOW_API_FALLBACK:
                results = paginate(
                    f"/sensors/{sid}/hours",
                    params={"datetime_from": iso(y_start), "datetime_to": iso(y_end)},
                    limit=MEAS_LIMIT,
                    max_pages=MAX_PAGES_MEAS,
                )
                print(f"[API BACKFILL] sensor={sid} rows_returned={len(results)}")

                records = []
                for r in results:
                    period = r.get("period") or {}
                    period_dt_to = period.get("datetimeTo") or {}
                    records.append(
                        {
                            "sensor_id": sid,
                            "parameter": normalize_param((r.get("parameter") or {}).get("name") if isinstance(r.get("parameter"), dict) else s_param),
                            "value": r.get("value"),
                            "datetime_utc": period_dt_to.get("utc"),
                            "datetime_local": period_dt_to.get("local"),
                        }
                    )
                df_chunk = pd.DataFrame.from_records(records)

            if df_chunk.empty:
                print(f"[CHUNK EMPTY] sensor={sid} from={iso(y_start)} to={iso(y_end)} returned no rows")
                sleep_time_aws()
                continue

            df_chunk["datetime_utc"] = pd.to_datetime(df_chunk["datetime_utc"], errors="coerce", utc=True)
            df_chunk["parameter"] = df_chunk["parameter"].map(normalize_param)

            for col, val in {
                "location_id": s_meta.get("location_id"),
                "location_name": s_meta.get("location_name"),
                "location_lat": s_meta.get("location_lat"),
                "location_lon": s_meta.get("location_lon"),
                "sensor_name": s_meta.get("sensor_name"),
                "units": s_meta.get("units"),
            }.items():
                if col not in df_chunk.columns:
                    df_chunk[col] = val

            df_chunk = df_chunk.drop_duplicates(subset=["sensor_id", "parameter", "datetime_utc", "value"]).copy()
            df_chunk = df_chunk.sort_values(["sensor_id", "datetime_utc"]).reset_index(drop=True)

            df_chunk["lag_1h"] = df_chunk.groupby("sensor_id")["value"].shift(1)
            df_chunk["lag_6h"] = df_chunk.groupby("sensor_id")["value"].shift(6)
            df_chunk["lag_12h"] = df_chunk.groupby("sensor_id")["value"].shift(12)
            df_chunk["lag_24h"] = df_chunk.groupby("sensor_id")["value"].shift(24)

            lag_lookup = df_chunk.set_index(["sensor_id", "datetime_utc"])["lag_1h"].to_dict()
            neighbor_avg = []
            neighbor_weighted = []

            row_sensor_id = sid
            for row in df_chunk.itertuples(index=False):
                row_sensor_id = row.sensor_id
                dt = row.datetime_utc
                neighs = neighbors_map.get(row_sensor_id, [])

                vals = []
                weighted_vals = []
                weights = []

                for nid, dist in neighs:
                    val = lag_lookup.get((nid, dt))
                    if val is None or pd.isna(val):
                        continue
                    vals.append(val)
                    w = 1.0 / (dist + 1e-6)
                    weighted_vals.append(val * w)
                    weights.append(w)

                neighbor_avg.append(np.mean(vals) if vals else np.nan)
                neighbor_weighted.append(sum(weighted_vals) / sum(weights) if weights else np.nan)

            df_chunk["neighbor_avg_1h"] = neighbor_avg
            df_chunk["neighbor_weighted_1h"] = neighbor_weighted

            valid_times = df_chunk["datetime_utc"].dropna()
            if not valid_times.empty:
                update_minmax(row_sensor_id, valid_times.min())
                update_minmax(row_sensor_id, valid_times.max())

            df_big_out = df_chunk.copy()
            df_big_out["datetime_utc"] = df_big_out["datetime_utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

            append_rows_to_csv(BIG_CSV_PATH, BIG_HEADER, df_big_out[BIG_HEADER].values.tolist())
            write_partition_csv(df_big_out, PART_DIR)

            chunk_label = f"sensor_{row_sensor_id}_{y_start.strftime('%Y%m%dT%H%M%S')}_{cend.strftime('%Y%m%dT%H%M%S')}"
            write_parquet_chunk(df_chunk, PARQUET_FULL_DIR, chunk_label)

        except Exception as e:
            failures.append(
                {
                    "sensor_id": sid,
                    "parameter": s_param,
                    "chunk_from": iso(y_start),
                    "chunk_to": iso(y_end),
                    "error": str(e),
                }
            )
            print(f"[CHUNK FAILED] sensor={sid} parameter={s_param} from={iso(y_start)} to={iso(y_end)} error={e}")

        sleep_time_aws()

    sleep_time_api()

if failures:
    pd.DataFrame(failures).to_csv(FAIL_CSV_PATH, index=False)

flush_cache()

print("-" * 60)
print("FAILURE SUMMARY")
print("-" * 60)
print(f"Total failures: {len(failures)}")
print(f"Cache hits: {CACHE_HITS}")
if failures:
    print(pd.DataFrame(failures).head(10).to_string(index=False))
print("-" * 60)

print(f"Done. Big file: {BIG_CSV_PATH}")

[SENSOR 1/6] sensor_id=24000
parameter     : pm25
location_id   : 8236
location_name : North Holywood
[YEAR START] sensor=24000 from=2026-01-01T00:00:00Z to=2026-03-24T21:50:07Z
[CHUNK START] sensor=24000 from=2023-03-25T21:50:07Z to=2023-04-24T21:50:07Z
[AWS BACKFILL] sensor=24000 rows=682
[CHUNK FAILED] sensor=24000 parameter=pm25 from=2023-03-25T21:50:07Z to=2023-04-24T21:50:07Z error=name 'cend' is not defined
[CHUNK START] sensor=24000 from=2023-04-24T21:50:07Z to=2023-05-24T21:50:07Z
[AWS BACKFILL] sensor=24000 rows=398
[CHUNK FAILED] sensor=24000 parameter=pm25 from=2023-04-24T21:50:07Z to=2023-05-24T21:50:07Z error=name 'cend' is not defined
[CHUNK START] sensor=24000 from=2023-05-24T21:50:07Z to=2023-06-23T21:50:07Z


KeyboardInterrupt: 

In [ ]:
print("\n" + "="*70)
print("TIMESTAMP DIAGNOSTIC CHECK")
print("="*70)

diag = pd.read_csv(BIG_CSV_PATH, nrows=5000)

print("Raw datetime_utc sample:")
print(diag["datetime_utc"].head(10).to_list())

diag["datetime_utc_parsed"] = pd.to_datetime(diag["datetime_utc"], errors="coerce", utc=True)

print("\nParsed datetime sample:")
print(diag[["datetime_utc", "datetime_utc_parsed"]].head(10).to_string(index=False))

print("\nCounts:")
print("rows:", len(diag))
print("non-null raw datetime_utc:", diag["datetime_utc"].notna().sum())
print("valid parsed datetime_utc:", diag["datetime_utc_parsed"].notna().sum())
print("invalid parsed datetime_utc:", diag["datetime_utc_parsed"].isna().sum())
print("="*70)

In [ ]:
print("\n" + "="*70)
print("DIAGNOSTICS")
print("="*70)
print(f"Failures                    : {len(failures)}")
print(f"sensor_time_min entries     : {len(sensor_time_min)}")
print(f"sensor_time_max entries     : {len(sensor_time_max)}")

all_mins = [v for v in sensor_time_min.values() if v is not None]
all_maxs = [v for v in sensor_time_max.values() if v is not None]

print(f"Valid mins collected        : {len(all_mins)}")
print(f"Valid maxs collected        : {len(all_maxs)}")
print(f"Requests total              : {REQUEST_COUNT}")
print(f"Rate limit hits             : {RATE_LIMIT_COUNT}")
print(f"Retry errors                : {RETRYABLE_ERROR_COUNT}")
print(f"HTTP errors                 : {HTTP_ERROR_COUNT}")
print(f"Big CSV exists              : {os.path.exists(BIG_CSV_PATH)}")
print(f"Big CSV size (bytes)        : {os.path.getsize(BIG_CSV_PATH) if os.path.exists(BIG_CSV_PATH) else 'missing'}")

try:
    sample_big = pd.read_csv(BIG_CSV_PATH, nrows=10)
    print("\nSample BIG_CSV rows:")
    print(sample_big.to_string(index=False))
except Exception as e:
    print("\nCould not read BIG_CSV sample:", e)

if failures:
    print("\nFailure sample:")
    print(pd.DataFrame(failures).head(10).to_string(index=False))
print("="*70 + "\n")

split_skipped = False
global_cutoff = None

if not all_mins or not all_maxs:
    print("WARNING: No valid timestamps found. No Split.")
    split_skipped = True
else:
    global_min = min(all_mins)
    global_max = max(all_maxs)

    if global_max <= global_min:
        print("WARNING: Invalid global time range. No Split.")
        split_skipped = True
    else:
        global_cutoff = global_min + (global_max - global_min) * 0.8
        print(f"Global cutoff: {global_cutoff}")

# **Test/Train Split**

In [ ]:
train_count = 0
test_count = 0
bad_count = 0

TRAIN_BIG = os.path.join(OUT_DIR, "train_long.csv")
TEST_BIG = os.path.join(OUT_DIR, "test_long.csv")

with open(TRAIN_BIG, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(BIG_HEADER)

with open(TEST_BIG, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(BIG_HEADER)

with open(BAD_ROWS_PATH, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(BIG_HEADER)

In [ ]:
if not split_skipped:
    for chunk_idx, chunk in enumerate(pd.read_csv(BIG_CSV_PATH, chunksize=200_000), start=1):
        print(f"[SPLIT] processing big CSV chunk {chunk_idx}")

        chunk["datetime_utc_parsed"] = pd.to_datetime(chunk["datetime_utc"], errors="coerce", utc=True)
        chunk["parameter"] = chunk["parameter"].map(normalize_param)

        chunk = chunk.drop_duplicates(
            subset=["sensor_id", "parameter", "datetime_utc", "value"]
        ).copy()

        train_rows = []
        test_rows = []
        bad_rows = []

        for row in chunk.itertuples(index=False):
            dtp = row.datetime_utc_parsed

            packed_row = [
                row.sensor_id, row.parameter, row.value, row.datetime_utc, row.datetime_local,
                row.location_id, row.location_name, row.location_lat, row.location_lon,
                row.sensor_name, row.units, row.lag_1h, row.lag_6h, row.lag_12h, row.lag_24h,
                row.neighbor_avg_1h, row.neighbor_weighted_1h
            ]

            if pd.isna(dtp):
                bad_rows.append(packed_row)
                continue

            if dtp <= global_cutoff:
                train_rows.append(packed_row)
            else:
                test_rows.append(packed_row)

        print(f"[SPLIT] chunk={chunk_idx} train={len(train_rows)} test={len(test_rows)} bad={len(bad_rows)}")

        if train_rows:
            append_rows_to_csv(TRAIN_BIG, BIG_HEADER, train_rows)
            train_count += len(train_rows)

            df_train = pd.DataFrame(train_rows, columns=BIG_HEADER)
            df_train["datetime_utc"] = pd.to_datetime(df_train["datetime_utc"], errors="coerce", utc=True)
            write_partition_csv(df_train, TRAIN_DIR)
            write_parquet_chunk(df_train, PARQUET_TRAIN_DIR, f"train_chunk_{chunk_idx:04d}")

        if test_rows:
            append_rows_to_csv(TEST_BIG, BIG_HEADER, test_rows)
            test_count += len(test_rows)

            df_test = pd.DataFrame(test_rows, columns=BIG_HEADER)
            df_test["datetime_utc"] = pd.to_datetime(df_test["datetime_utc"], errors="coerce", utc=True)
            write_partition_csv(df_test, TEST_DIR)
            write_parquet_chunk(df_test, PARQUET_TEST_DIR, f"test_chunk_{chunk_idx:04d}")

        if bad_rows:
            append_rows_to_csv(BAD_ROWS_PATH, BIG_HEADER, bad_rows)
            bad_count += len(bad_rows)

            df_bad = pd.DataFrame(bad_rows, columns=BIG_HEADER)
            df_bad["datetime_utc"] = pd.to_datetime(df_bad["datetime_utc"], errors="coerce", utc=True)
            write_parquet_chunk(df_bad, PARQUET_BAD_DIR, f"bad_chunk_{chunk_idx:04d}")
else:
    print("No Split Made")

print("\n" + "-"*60)
print("SPLIT SUMMARY")
print("-"*60)
print(f"Train rows: {train_count:,}")
print(f"Test rows : {test_count:,}")
print(f"Bad rows  : {bad_count:,}")
print(f"Train big : {TRAIN_BIG}")
print(f"Test big  : {TEST_BIG}")
print(f"Full part : {PART_DIR}")
print(f"Train part: {TRAIN_DIR}")
print(f"Test part : {TEST_DIR}")
print("-"*60 + "\n")

# **Summary**

In [ ]:
summary = {
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "area": {"lat": LAT, "lon": LON, "radius_m": RADIUS_M},
    "days_back": DAYS,
    "chunk_days": CHUNK_DAYS,
    "params_to_pull": params_to_pull,
    "split_skipped": split_skipped,
    "global_cutoff": str(global_cutoff) if global_cutoff is not None else None,
    "counts": {
        "locations": len(locations),
        "sensors_total": int(df_sensors.shape[0]),
        "sensors_used": len(sensor_ids),
        "failures": len(failures),
        "requests_total": REQUEST_COUNT,
        "rate_limit_hits": RATE_LIMIT_COUNT,
        "retryable_errors": RETRYABLE_ERROR_COUNT,
        "http_errors": HTTP_ERROR_COUNT,
        "train_rows": train_count,
        "test_rows": test_count,
        "bad_rows": bad_count,
    },
    "files": {
        "big_csv": BIG_CSV_PATH,
        "sensors_csv": SENSORS_CSV_PATH,
        "failures_csv": FAIL_CSV_PATH if os.path.exists(FAIL_CSV_PATH) else None,
        "bad_rows_csv": BAD_ROWS_PATH,
        "train_big": TRAIN_BIG,
        "test_big": TEST_BIG,
        "parquet_full_dir": PARQUET_FULL_DIR,
        "parquet_train_dir": PARQUET_TRAIN_DIR,
        "parquet_test_dir": PARQUET_TEST_DIR,
        "parquet_bad_dir": PARQUET_BAD_DIR,
        "sensor_map": ARC_SENSOR_MAP,
        "distance_matrix_csv": DIST_PATH
    }
}

sum_path = os.path.join(OUT_DIR, "run_summary.json")
with open(sum_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Saved summary: {sum_path}")

# **ArcGIS Map**

In [ ]:
from arcgis.gis import GIS
from arcgis.geometry import Point
from arcgis.geometry.functions import buffer, LengthUnits
from arcgis.map.popups import PopupInfo
from arcgis.map.symbols import (
    SimpleMarkerSymbolEsriSMS,
    SimpleLineSymbolEsriSLS,
    SimpleFillSymbolEsriSFS,
    SimpleFillSymbolStyle,
    SimpleMarkerSymbolStyle,
)
from IPython.display import display, HTML
import pandas as pd

gis = GIS(api_key=ARC_KEY)

df_map = (
    df_sensors_use
    .dropna(subset=["location_lat", "location_lon"])
    .copy()
)

if df_map.empty:
    raise ValueError("No sensor coordinates available to draw on the ArcGIS map.")

df_map["sensor_id"] = df_map["sensor_id"].astype(str)
df_map["sensor_name"] = df_map["sensor_name"].fillna("unknown").astype(str)
df_map["parameter"] = df_map["parameter"].fillna("unknown").astype(str)
df_map["location_name"] = df_map["location_name"].fillna("unknown").astype(str)
df_map["units"] = df_map["units"].fillna("").astype(str)

df_map = df_map.drop_duplicates(subset=["sensor_id", "location_lat", "location_lon"]).reset_index(drop=True)

def start_zoom_from_radius(radius_m: int) -> int:
    return 11

def sensor_symbol(param: str):
    p = (param or "").lower()

    if p in {"pm25", "pm2.5"}:
        return SimpleMarkerSymbolEsriSMS(
            style=SimpleMarkerSymbolStyle.esri_sms_circle,
            color=[34, 211, 238, 230],
            size=10,
            outline=SimpleLineSymbolEsriSLS(color=[15, 23, 42, 220], width=1.2),
        )
    elif p == "pm10":
        return SimpleMarkerSymbolEsriSMS(
            style=SimpleMarkerSymbolStyle.esri_sms_circle,
            color=[251, 191, 36, 230],
            size=10,
            outline=SimpleLineSymbolEsriSLS(color=[15, 23, 42, 220], width=1.2),
        )
    else:
        return SimpleMarkerSymbolEsriSMS(
            style=SimpleMarkerSymbolStyle.esri_sms_circle,
            color=[167, 139, 250, 230],
            size=10,
            outline=SimpleLineSymbolEsriSLS(color=[15, 23, 42, 220], width=1.2),
        )


def sensor_popup(row) -> PopupInfo:
    return PopupInfo(
        title=f"Sensor {row.sensor_id} · {str(row.parameter).upper()}",
        description=f"""
        <div style="font-family: Arial, sans-serif; font-size: 13px; line-height: 1.45;">
            <b>Sensor name:</b> {row.sensor_name}<br>
            <b>Location:</b> {row.location_name}<br>
            <b>Parameter:</b> {row.parameter}<br>
            <b>Units:</b> {row.units if row.units else "n/a"}<br>
            <b>Latitude:</b> {float(row.location_lat):.5f}<br>
            <b>Longitude:</b> {float(row.location_lon):.5f}
        </div>
        """
    )


radius_symbol = SimpleFillSymbolEsriSFS(
    style=SimpleFillSymbolStyle.esri_sfs_solid,
    color=[56, 189, 248, 18],
    outline=SimpleLineSymbolEsriSLS(color=[56, 189, 248, 235], width=2.4),
)

center_symbol = SimpleMarkerSymbolEsriSMS(
    style=SimpleMarkerSymbolStyle.esri_sms_cross,
    color=[255, 255, 255, 255],
    size=14,
    outline=SimpleLineSymbolEsriSLS(color=[14, 165, 233, 255], width=1.5),
)

def popup_html(row) -> str:
    return f"""
    <div style="
        font-family: Inter, Arial, sans-serif;
        font-size: 13px;
        line-height: 1.45;
        color: #111827;
        min-width: 220px;
    ">
        <div style="font-size: 14px; font-weight: 700; margin-bottom: 6px;">
            Sensor {row.sensor_id}
        </div>
        <div><b>Sensor name:</b> {row.sensor_name}</div>
        <div><b>Location:</b> {row.location_name}</div>
        <div><b>Parameter:</b> {row.parameter}</div>
        <div><b>Units:</b> {row.units if row.units else "n/a"}</div>
        <div><b>Latitude:</b> {float(row.location_lat):.5f}</div>
        <div><b>Longitude:</b> {float(row.location_lon):.5f}</div>
    </div>
    """

center_pt = Point({
    "x": float(LON),
    "y": float(LAT),
    "spatialReference": {"wkid": 4326}
})

radius_polygon = buffer(
    geometries=[center_pt],
    in_sr=4326,
    distances=[RADIUS_M],
    unit=LengthUnits.METER,
    out_sr=4326,
    geodesic=True,
    gis=gis,
    future=False
)[0]

#make Map
ARC_MAP = gis.map()
ARC_MAP.center = [LAT,LON]
ARC_MAP.zoom = start_zoom_from_radius(RADIUS_M)

try:
    ARC_MAP.basemap.basemap = "dark-gray-vector"
except Exception:
    pass

ARC_MAP.content.draw(
    shape=radius_polygon,
    symbol=radius_symbol,
    popup=PopupInfo(
        title="Model Search Radius",
        description=f"""
        <div style="font-family: Arial, sans-serif; font-size: 13px; line-height: 1.45;">
            <b>Center:</b> ({LAT:.5f}, {LON:.5f})<br>
            <b>Radius:</b> {RADIUS_M:,} meters<br>
            <b>Radius:</b> {RADIUS_M / 1000:.2f} km
        </div>
        """
    ),
    attributes={
        "name": "search_radius",
        "radius_m": RADIUS_M
    }
)

ARC_MAP.content.draw(
    shape=center_pt,
    symbol=center_symbol,
    popup=PopupInfo(
        title="Area Center",
        description=f"""
        <div style="font-family: Arial, sans-serif; font-size: 13px; line-height: 1.45;">
            <b>Latitude:</b> {LAT:.5f}<br>
            <b>Longitude:</b> {LON:.5f}
        </div>
        """
    ),
    attributes={"name": "center_point"}
)


for row in df_map.sort_values(["parameter", "sensor_id"]).itertuples(index=False):
    pt = Point({
        "x": float(row.location_lon),
        "y": float(row.location_lat),
        "spatialReference": {"wkid": 4326}
    })

    ARC_MAP.content.draw(
        shape=pt,
        symbol=sensor_symbol(row.parameter),
        popup=sensor_popup(row),
        attributes={
            "sensor_id": row.sensor_id,
            "parameter": row.parameter,
            "location_name": row.location_name
        }
    )


ARC_MAP.export_to_html(ARC_SENSOR_MAP, title="OpenAQ Sensor Map")
print(f"Saved ArcGIS map: {ARC_SENSOR_MAP}")


counts = df_map["parameter"].value_counts(dropna=False).to_dict()
pm25_count = counts.get("pm25", 0) + counts.get("pm2.5", 0)
pm10_count = counts.get("pm10", 0)
other_count = len(df_map) - pm25_count - pm10_count

legend_html = f"""
<div style="
    font-family: Inter, Arial, sans-serif;
    background: linear-gradient(135deg, #0f172a 0%, #111827 100%);
    color: #e5e7eb;
    border-radius: 16px;
    padding: 14px 18px;
    margin: 8px 0 14px 0;
    box-shadow: 0 10px 24px rgba(0,0,0,0.22);
    max-width: 540px;
">
    <div style="font-size: 18px; font-weight: 700; margin-bottom: 8px;">
        OpenAQ Sensor Area
    </div>
    <div style="font-size: 13px; color: #cbd5e1; margin-bottom: 12px;">
        Center: ({LAT:.5f}, {LON:.5f}) &nbsp; | &nbsp; Radius: {RADIUS_M:,} m
    </div>

    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 8px 16px; font-size: 13px;">
        <div><span style="display:inline-block;width:10px;height:10px;background:#22d3ee;border-radius:50%;margin-right:8px;"></span>PM2.5 sensors: <b>{pm25_count}</b></div>
        <div><span style="display:inline-block;width:10px;height:10px;background:#fbbf24;border-radius:50%;margin-right:8px;"></span>PM10 sensors: <b>{pm10_count}</b></div>
        <div><span style="display:inline-block;width:10px;height:10px;background:#a78bfa;border-radius:50%;margin-right:8px;"></span>Other sensors: <b>{other_count}</b></div>
        <div><span style="display:inline-block;width:10px;height:10px;background:#38bdf8;border-radius:50%;margin-right:8px;"></span>Total mapped: <b>{len(df_map)}</b></div>
    </div>
</div>
"""

display(HTML(legend_html))
ARC_MAP

In [ ]:
print("Main output folder:", OUT_DIR)
print("Big CSV:", BIG_CSV_PATH)
print("Sensors CSV:", SENSORS_CSV_PATH)
print("Train CSV:", TRAIN_BIG)
print("Test CSV:", TEST_BIG)
print("Summary JSON:", sum_path)
print("ArcGIS Map:", ARC_SENSOR_MAP)

In [ ]:
# ==============================================================
# NOAA / NCEI HISTORICAL WEATHER DATA INTEGRATION  (Vineet's Part)
# ==============================================================

!pip install -q requests pandas tqdm matplotlib

import requests
import os
import time
import pandas as pd
import numpy as np
from datetime import timedelta
from tqdm import tqdm
import matplotlib.pyplot as plt
from google.colab import userdata

# ------------- CONFIGURATION -------------
# 🔑 Paste your NOAA NCEI API token here:
NCEI_TOKEN= userdata.get("NCEI_TOKEN")
if not NCEI_TOKEN:
    raise ValueError("Missing NCEI_TOKEN API Key. Add it in Colab secrets as 'NCEI_TOKEN'")

if NCEI_TOKEN == "PASTE_YOUR_NOAA_TOKEN_HERE":
    raise ValueError("⚠️ Please replace 'PASTE_YOUR_NOAA_TOKEN_HERE' with your actual token from [ncdc.noaa.gov](https://www.ncdc.noaa.gov/cdo-web/token)")

HEADERS_NCEI = {"token": NCEI_TOKEN}
DATASET_ID = "ISD"  # Integrated Surface Dataset (hourly data)
DATA_VARS = ["WSF2", "WDF2", "TMP", "RHAV"]  # wind speed, direction, temperature, humidity

# --- Point these paths to your existing OpenAQ outputs ---
BIG_CSV_PATH = os.path.join(OUT_DIR, "all_measurements_long.csv")
OUT_DIR = OUT_DIR  # reuse your existing output folder variable

# ------------- Load existing OpenAQ sensor data -------------
if not os.path.exists(BIG_CSV_PATH):
    raise FileNotFoundError(f"OpenAQ dataset not found: {BIG_CSV_PATH}. Run the OpenAQ pipeline first.")

sensor_df = pd.read_csv(BIG_CSV_PATH)
sensor_df["datetime_utc"] = pd.to_datetime(sensor_df["datetime_utc"], utc=True)
print(f"✅ Loaded {len(sensor_df):,} sensor readings.")

# ------------- Helper to query NOAA NCEI -------------
def get_ncei_by_day(lat, lon, start_date, end_date, dataset=DATASET_ID, vars_list=None, radius_km=RADIUS_M/1000):
    """Fetch NOAA NCEI observation data near given coordinates and date range."""
    base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
    params = {
        "datasetid": dataset,
        "startdate": start_date,
        "enddate": end_date,
        "units": "metric",
        "latitude": lat,
        "longitude": lon,
        "radius": radius_km,
        "limit": 1000
    }
    if vars_list:
        params["datatypeid"] = vars_list
    try:
        r = requests.get(base_url, headers=HEADERS_NCEI, params=params, timeout=60)
        if r.status_code != 200:
            print(f"[WARN] {r.status_code} for {lat},{lon}: {r.text[:120]}")
            return pd.DataFrame()
        results = r.json().get("results", [])
        if not results:
            return pd.DataFrame()
        df = pd.DataFrame(results)
        df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
        return df
    except Exception as e:
        print(f"[ERROR] for {lat},{lon}: {e}")
        return pd.DataFrame()

# ------------- Sampling for testing -------------
SAMPLE_SIZE = min(5, len(sensor_df))  # start small to avoid long run times
sensor_subset = sensor_df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"🔹 Sampling {SAMPLE_SIZE} sensor readings for NCEI test...")

# ------------- Fetch weather data -------------
merged_records = []

for _, row in tqdm(sensor_subset.iterrows(), total=len(sensor_subset)):
    lat, lon = row["location_lat"], row["location_lon"]
    dt = row["datetime_utc"]

    start_date = dt.strftime("%Y-%m-%d")
    end_date   = (dt + timedelta(days=1)).strftime("%Y-%m-%d")

    df_weather = get_ncei_by_day(lat, lon, start_date, end_date, vars_list=DATA_VARS)

    if not df_weather.empty:
        means = df_weather.groupby("datatype")["value"].mean().to_dict()
        merged_records.append({
            "sensor_id":    row["sensor_id"],
            "datetime_utc": dt,
            "wind_speed":   means.get("WSF2"),
            "wind_dir":     means.get("WDF2"),
            "temperature":  means.get("TMP"),
            "humidity":     means.get("RHAV")
        })

    # pause slightly to respect API limits
    time.sleep(1)

# ------------- Merge and save -------------
if merged_records:
    weather_df = pd.DataFrame(merged_records)
    merged_df = sensor_df.merge(weather_df, on=["sensor_id", "datetime_utc"], how="left")

    merged_path = os.path.join(OUT_DIR, "merged_with_ncei.csv")
    merged_df.to_csv(merged_path, index=False)
    print(f"\n✅ Merged sensor + weather data saved to:\n   {merged_path}")
    print(f"  Total rows: {len(merged_df):,}")

    # quick peek
    print(merged_df.head())
else:
    print("\n⚠️ No NCEI data returned — check token or radius")

# ------------- Optional: quick visualization -------------
try:
    subset_plot = merged_df.dropna(subset=["wind_speed", "wind_dir"]).sample(
        n=min(150, len(merged_df)), random_state=42
    )
    u = subset_plot["wind_speed"] * np.cos(np.deg2rad(subset_plot["wind_dir"]))
    v = subset_plot["wind_speed"] * np.sin(np.deg2rad(subset_plot["wind_dir"]))

    plt.figure(figsize=(9,6))
    plt.quiver(subset_plot["location_lon"], subset_plot["location_lat"], u, v,
               color="blue", scale=50, alpha=0.6, label="Wind Vectors")
    plt.scatter(subset_plot["location_lon"], subset_plot["location_lat"],
                c=subset_plot["value"], cmap="Reds", s=40, label="PM2.5 Level")
    plt.legend()
    plt.title("Wind Flow vs PM2.5 Concentration (sample overlay)")
    plt.xlabel("Longitude"); plt.ylabel("Latitude")
    plt.show()
except Exception as e:
    print("Plot skipped:", e)
